#Session 8 | Homework

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, sum

In [8]:
spark = (
    SparkSession.builder
    .appName("Session08Part02")
    .master("local[*]")
    .getOrCreate()
)

In [11]:
orders = [
    (1, "Athens", "Laptop", "Electronics", 5, 1200.00),
    (2, "Athens", "Mouse", "Electronics", 40, 25.00),
    (3, "Thessaloniki", "Desk", "Furniture", 10, 240.00),
    (4, "Patras", "Chair", "Furniture", 25, 85.00),
    (5, "Athens", "Notebook", "Office", 47, 3.50),
    (6, "Heraklion", "Monitor", "Electronics", 100, 310.00),
    (7, "Patras", "Pen Pack", "Office", 69, 6.00),
    (8, "Thessaloniki", "Keyboard", "Electronics", 83, 75.00),
    (9, "Athens", "Desk Lamp", "Furniture", 35, 32.00),
    (10, "Heraklion", "Notebook", "Office", 72, 3.25),
    (11, "Athens", "Selve", "Furniture", 38, 10.00),
    (12, "Patras", "Cable", "Electronics", 26, 5.00),
    (13, "Heraklion", "Pen", "Office", 95, 0.50),
    (14, "Thessaloniki", "Door", "Furniture", 5, 350.00),
    (15, "Patras", "TV", "Electronics", 4, 725.00)
]

columns = ["order_id", "city", "product", "category", "quantity", "unit_price"]

orders_df = spark.createDataFrame(orders, columns)
orders_df.show()

+--------+------------+---------+-----------+--------+----------+
|order_id|        city|  product|   category|quantity|unit_price|
+--------+------------+---------+-----------+--------+----------+
|       1|      Athens|   Laptop|Electronics|       5|    1200.0|
|       2|      Athens|    Mouse|Electronics|      40|      25.0|
|       3|Thessaloniki|     Desk|  Furniture|      10|     240.0|
|       4|      Patras|    Chair|  Furniture|      25|      85.0|
|       5|      Athens| Notebook|     Office|      47|       3.5|
|       6|   Heraklion|  Monitor|Electronics|     100|     310.0|
|       7|      Patras| Pen Pack|     Office|      69|       6.0|
|       8|Thessaloniki| Keyboard|Electronics|      83|      75.0|
|       9|      Athens|Desk Lamp|  Furniture|      35|      32.0|
|      10|   Heraklion| Notebook|     Office|      72|      3.25|
|      11|      Athens|    Selve|  Furniture|      38|      10.0|
|      12|      Patras|    Cable|Electronics|      26|       5.0|
|      13|

Revenue Column:

In [12]:
orders_df = orders_df.withColumn("revenue", col("quantity") * col("unit_price"))

orders_df.show()

+--------+------------+---------+-----------+--------+----------+-------+
|order_id|        city|  product|   category|quantity|unit_price|revenue|
+--------+------------+---------+-----------+--------+----------+-------+
|       1|      Athens|   Laptop|Electronics|       5|    1200.0| 6000.0|
|       2|      Athens|    Mouse|Electronics|      40|      25.0| 1000.0|
|       3|Thessaloniki|     Desk|  Furniture|      10|     240.0| 2400.0|
|       4|      Patras|    Chair|  Furniture|      25|      85.0| 2125.0|
|       5|      Athens| Notebook|     Office|      47|       3.5|  164.5|
|       6|   Heraklion|  Monitor|Electronics|     100|     310.0|31000.0|
|       7|      Patras| Pen Pack|     Office|      69|       6.0|  414.0|
|       8|Thessaloniki| Keyboard|Electronics|      83|      75.0| 6225.0|
|       9|      Athens|Desk Lamp|  Furniture|      35|      32.0| 1120.0|
|      10|   Heraklion| Notebook|     Office|      72|      3.25|  234.0|
|      11|      Athens|    Selve|  Fur

Register a temporary SQL view

In [13]:
orders_df.createOrReplaceTempView("orders")

## SQL Queries

Which city has the highest total revenue?

In [14]:
spark.sql("""
    SELECT city, SUM(revenue) AS total_revenue
    FROM orders
    GROUP BY city
    ORDER BY total_revenue DESC
""").show()

+------------+-------------+
|        city|total_revenue|
+------------+-------------+
|   Heraklion|      31281.5|
|Thessaloniki|      10375.0|
|      Athens|       8664.5|
|      Patras|       5569.0|
+------------+-------------+



Which category has the most orders?

In [15]:
spark.sql("""
    SELECT category, SUM(quantity) AS total_orders
    FROM orders
    GROUP BY category
    ORDER BY total_orders DESC
""").show()

+-----------+------------+
|   category|total_orders|
+-----------+------------+
|     Office|         283|
|Electronics|         258|
|  Furniture|         113|
+-----------+------------+



Which products have revenue greater than 2100?

In [17]:
spark.sql("""
    SELECT *
    FROM orders
    WHERE revenue >= 2100
""").show()

+--------+------------+--------+-----------+--------+----------+-------+
|order_id|        city| product|   category|quantity|unit_price|revenue|
+--------+------------+--------+-----------+--------+----------+-------+
|       1|      Athens|  Laptop|Electronics|       5|    1200.0| 6000.0|
|       3|Thessaloniki|    Desk|  Furniture|      10|     240.0| 2400.0|
|       4|      Patras|   Chair|  Furniture|      25|      85.0| 2125.0|
|       6|   Heraklion| Monitor|Electronics|     100|     310.0|31000.0|
|       8|Thessaloniki|Keyboard|Electronics|      83|      75.0| 6225.0|
|      15|      Patras|      TV|Electronics|       4|     725.0| 2900.0|
+--------+------------+--------+-----------+--------+----------+-------+



What is the average unit price per category?

In [18]:
spark.sql("""
    SELECT category, SUM(unit_price)/COUNT(category) AS avg_price_per_unit
    FROM orders
    GROUP BY category
""").show()

+-----------+------------------+
|   category|avg_price_per_unit|
+-----------+------------------+
|     Office|            3.3125|
|Electronics|             390.0|
|  Furniture|             143.4|
+-----------+------------------+



Stop Spark at the end

In [19]:
spark.stop()